### GPT-oss

In [ ]:
import json
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Set fonts.
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


model_name = "gpt-oss-120b"
# Two data directories.
pg19_dir = f"paper/results/gemini-2.0-flash-001/{model_name}-medium"
relevant_dir = f"paper/Analysis/Context_Type/results_relevant_contexts/{model_name}-medium"

def load_data(base_dir):
    """Load safety rate data from a directory"""
    # data[level][context_length] = safe_rate
    data = defaultdict(dict)
    
    if not os.path.exists(base_dir):
        print(f"Directory not found: {base_dir}")
        return data
    
    for filename in os.listdir(base_dir):
        if filename.startswith('eval_level_'):
            # Parse filename: eval_level_X_Yk_results_google-gemini-2.0-flash-lite-001.json
            match = re.match(r'eval_level_(\d+)_(\d+)k_results_google-gemini-2.0-flash-lite-001.json', filename)
            if match:
                level = int(match.group(1))
                context_length = int(match.group(2))
                
                filepath = os.path.join(base_dir, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        result = json.load(f)
                        stats = result.get('statistics', {})
                        total = stats.get('total', 0)
                        score_dist = stats.get('score_distribution', {})
                        # score=5 as unsafe
                        score_5_count = score_dist.get('5', 0)
                        
                        if total > 0:
                            safe_rate = (total - score_5_count) / total * 100
                            data[level][context_length] = safe_rate
                        else:
                            data[level][context_length] = 0.0
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
    
    return data

# Load data from both directories
pg19_data = load_data(pg19_dir)
relevant_data = load_data(relevant_dir)

# Print data overview
print("PG19 Data:")
for level in sorted(pg19_data.keys()):
    print(f"  Level {level}: {sorted(pg19_data[level].keys())}")

print("\nRelevant Context Data:")
for level in sorted(relevant_data.keys()):
    print(f"  Level {level}: {sorted(relevant_data[level].keys())}")

# Collect all context lengths
all_context_lengths_set = set()
for level_data in pg19_data.values():
    all_context_lengths_set.update(level_data.keys())
for level_data in relevant_data.values():
    all_context_lengths_set.update(level_data.keys())
all_context_lengths = sorted(list(all_context_lengths_set))

print(f"\nAll Context Lengths: {all_context_lengths}")


In [ ]:
# Figure 3: Heatmap comparison of differences.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Build the data matrix.
levels = [0, 1, 2, 3, 4]
ctx_lengths = all_context_lengths

# Heatmap 1: PG19 safety rate.
pg19_matrix = np.full((len(levels), len(ctx_lengths)), np.nan)
for i, level in enumerate(levels):
    for j, ctx in enumerate(ctx_lengths):
        if ctx in pg19_data.get(level, {}):
            pg19_matrix[i, j] = pg19_data[level][ctx]

# Heatmap 2: Relevant-context safety rate.
rel_matrix = np.full((len(levels), len(ctx_lengths)), np.nan)
for i, level in enumerate(levels):
    for j, ctx in enumerate(ctx_lengths):
        if ctx in relevant_data.get(level, {}):
            rel_matrix[i, j] = relevant_data[level][ctx]

# Heatmap 3: Difference (Relevant - PG19).
diff_matrix = rel_matrix - pg19_matrix

# Plot 1: PG19
im1 = axes[0].imshow(pg19_matrix, cmap='RdYlGn', aspect='auto', vmin=50, vmax=100)
axes[0].set_title('PG19 (Irrelevant) Safety Ratio (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Context Length', fontsize=11)
axes[0].set_ylabel('Level', fontsize=11)
axes[0].set_xticks(range(len(ctx_lengths)))
axes[0].set_xticklabels([f'{x}k' for x in ctx_lengths])
axes[0].set_yticks(range(len(levels)))
axes[0].set_yticklabels(levels)
# Add text annotations
for i in range(len(levels)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(pg19_matrix[i, j]):
            axes[0].text(j, i, f'{pg19_matrix[i, j]:.1f}', ha='center', va='center', 
                        fontsize=8, fontweight='bold', color='black')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Plot 2: Relevant
im2 = axes[1].imshow(rel_matrix, cmap='RdYlGn', aspect='auto', vmin=50, vmax=100)
axes[1].set_title('Relevant Context Safety Rate (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Context Length', fontsize=11)
axes[1].set_ylabel('Level', fontsize=11)
axes[1].set_xticks(range(len(ctx_lengths)))
axes[1].set_xticklabels([f'{x}k' for x in ctx_lengths])
axes[1].set_yticks(range(len(levels)))
axes[1].set_yticklabels(levels)
for i in range(len(levels)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(rel_matrix[i, j]):
            axes[1].text(j, i, f'{rel_matrix[i, j]:.1f}', ha='center', va='center', 
                        fontsize=8, fontweight='bold', color='black')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Plot 3: Difference
im3 = axes[2].imshow(diff_matrix, cmap='RdBu', aspect='auto', vmin=-20, vmax=20)
axes[2].set_title('Difference (Relevant - PG19) %', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Context Length', fontsize=11)
axes[2].set_ylabel('Level', fontsize=11)
axes[2].set_xticks(range(len(ctx_lengths)))
axes[2].set_xticklabels([f'{x}k' for x in ctx_lengths])
axes[2].set_yticks(range(len(levels)))
axes[2].set_yticklabels(levels)
for i in range(len(levels)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(diff_matrix[i, j]):
            val = diff_matrix[i, j]
            color = 'black'
            axes[2].text(j, i, f'{val:+.1f}', ha='center', va='center', 
                        fontsize=8, fontweight='bold', color=color)
plt.colorbar(im3, ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.savefig('context_comparison_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# 1x4 layout: Irrelevant heatmap, relevant heatmap, difference heatmap, and average bar chart.

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

# Basic configuration.
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
})

# Data preparation.
levels_to_use = [0, 1, 2, 3]
type_labels = ["base", "1", "2", "3"]
ctx_lengths = all_context_lengths

# Build matrices.
pg19_matrix = np.full((len(levels_to_use), len(ctx_lengths)), np.nan)
rel_matrix = np.full((len(levels_to_use), len(ctx_lengths)), np.nan)

for i, level in enumerate(levels_to_use):
    for j, ctx in enumerate(ctx_lengths):
        pg19_val = pg19_data.get(level, {}).get(ctx)
        rel_val = relevant_data.get(level, {}).get(ctx)
        if pg19_val is not None:
            pg19_matrix[i, j] = pg19_val
        if rel_val is not None:
            rel_matrix[i, j] = rel_val

diff_matrix = rel_matrix - pg19_matrix

# Create a 1x4 figure.
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# ========== Subplot 1: PG19 (irrelevant) heatmap ==========
ax = axes[0]
im1 = ax.imshow(pg19_matrix, cmap='RdYlGn', aspect='auto', vmin=30, vmax=100)
ax.set_title('PG19 (Irrelevant) %', fontsize=12, fontweight='bold')
ax.set_xlabel('Context Length', fontsize=11)
ax.set_ylabel('Type', fontsize=11)
ax.set_xticks(range(len(ctx_lengths)))
ax.set_xticklabels([f'{x}k' for x in ctx_lengths], fontsize=9)
ax.set_yticks(range(len(levels_to_use)))
ax.set_yticklabels([f'{t}' for t in type_labels], fontsize=9)
for i in range(len(levels_to_use)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(pg19_matrix[i, j]):
            val = pg19_matrix[i, j]
            text_color = 'white' if val < 50 else 'black'
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)
plt.colorbar(im1, ax=ax, shrink=0.8, pad=0.02)

# ========== Subplot 2: Relevant heatmap ==========
ax = axes[1]
im2 = ax.imshow(rel_matrix, cmap='RdYlGn', aspect='auto', vmin=30, vmax=100)
ax.set_title('Relevant Context %', fontsize=12, fontweight='bold')
ax.set_xlabel('Context Length', fontsize=11)
ax.set_xticks(range(len(ctx_lengths)))
ax.set_xticklabels([f'{x}k' for x in ctx_lengths], fontsize=9)
ax.set_yticks(range(len(levels_to_use)))
ax.set_yticklabels([f'{t}' for t in type_labels], fontsize=9)
for i in range(len(levels_to_use)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(rel_matrix[i, j]):
            val = rel_matrix[i, j]
            text_color = 'white' if val < 50 else 'black'
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)
plt.colorbar(im2, ax=ax, shrink=0.8, pad=0.02)

# ========== Subplot 3: Difference heatmap ==========
ax = axes[2]
norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=40)
im3 = ax.imshow(diff_matrix, cmap='RdBu', aspect='auto', norm=norm)
ax.set_title('Difference (Rel − PG19) %', fontsize=12, fontweight='bold')
ax.set_xlabel('Context Length', fontsize=11)
ax.set_xticks(range(len(ctx_lengths)))
ax.set_xticklabels([f'{x}k' for x in ctx_lengths], fontsize=9)
ax.set_yticks(range(len(levels_to_use)))
ax.set_yticklabels([f'{t}' for t in type_labels], fontsize=9)
for i in range(len(levels_to_use)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(diff_matrix[i, j]):
            val = diff_matrix[i, j]
            text_color = 'white' if abs(val) > 18 else 'black'
            ax.text(j, i, f'{val:+.0f}', ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)
plt.colorbar(im3, ax=ax, shrink=0.8, pad=0.02)

# ========== Subplot 4: Average safety rate (Types 2-4) ==========
ax = axes[3]
levels_for_avg = [1, 2, 3]  # Type 2, 3, 4
pg19_avg_by_ctx = {}
rel_avg_by_ctx = {}

for ctx in ctx_lengths:
    pg19_vals = [pg19_data[lvl].get(ctx) for lvl in levels_for_avg if pg19_data.get(lvl, {}).get(ctx) is not None]
    rel_vals = [relevant_data[lvl].get(ctx) for lvl in levels_for_avg if relevant_data.get(lvl, {}).get(ctx) is not None]
    if pg19_vals:
        pg19_avg_by_ctx[ctx] = np.mean(pg19_vals)
    if rel_vals:
        rel_avg_by_ctx[ctx] = np.mean(rel_vals)

x = list(pg19_avg_by_ctx.keys())
pg19_vals = [pg19_avg_by_ctx[ctx] for ctx in x]
rel_vals = [rel_avg_by_ctx.get(ctx, np.nan) for ctx in x]

width = 0.35
x_pos = np.arange(len(x))

ax.bar(x_pos - width/2, pg19_vals, width, label='PG19 (Irrelevant)', color='#2E86AB', edgecolor='white')
ax.bar(x_pos + width/2, rel_vals, width, label='Relevant Context', color='#E94F37', edgecolor='white')

ax.set_xlabel('Context Length', fontsize=11)
ax.set_ylabel('Avg Safety Ratio (%)', fontsize=11)
ax.set_title('Average (Type 1−3)', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{ctx}k' for ctx in x], fontsize=9)
ax.set_ylim(30, 105)
ax.grid(True, alpha=0.3, linestyle='--', axis='y')
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.savefig(f'context_comparison_{model_name}.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics.
print("\n" + "=" * 50)
print(f"{'Context':<10} {'PG19':>10} {'Relevant':>10} {'Diff':>10}")
for ctx in ctx_lengths:
    p_v = pg19_avg_by_ctx.get(ctx, 0)
    r_v = rel_avg_by_ctx.get(ctx, 0)
    print(f"{ctx}k{'':<8} {p_v:>10.1f} {r_v:>10.1f} {r_v-p_v:>+10.1f}")

In [ ]:
# Combined figure: Difference heatmap on the left and average safety ratio on the right.
# Use levels 0, 1, 2, and 3, displayed as Types 1, 2, 3, and 4.

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

# Increase the global font size.
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
})

# Use levels 0, 1, 2, and 3, displayed as Types 1, 2, 3, and 4.
levels_to_use = [0, 1, 2, 3]  # Levels in the actual data.
type_labels = [1, 2, 3, 4]    # Display labels.
ctx_lengths = all_context_lengths

# Build the difference matrix.
diff_matrix = np.full((len(levels_to_use), len(ctx_lengths)), np.nan)
for i, level in enumerate(levels_to_use):
    for j, ctx in enumerate(ctx_lengths):
        pg19_val = pg19_data.get(level, {}).get(ctx)
        rel_val = relevant_data.get(level, {}).get(ctx)
        if pg19_val is not None and rel_val is not None:
            diff_matrix[i, j] = rel_val - pg19_val

# Create the figure.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ========== Left plot: Difference heatmap ==========
ax = axes[0]
# Use TwoSlopeNorm centered at 0, with red for negative values and blue for positive values.
norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=40)
im = ax.imshow(diff_matrix, cmap='RdBu', aspect='auto', norm=norm)

ax.set_title('Difference (Relevant − PG19) %', fontsize=16, fontweight='bold', pad=10)
ax.set_xlabel('Context Length', fontsize=14, fontweight='bold')
ax.set_ylabel('Type', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(ctx_lengths)))
ax.set_xticklabels([f'{x}k' for x in ctx_lengths], fontsize=12)
ax.set_yticks(range(len(levels_to_use)))
ax.set_yticklabels([f'{t}' for t in type_labels], fontsize=12)

# Add numeric labels.
for i in range(len(levels_to_use)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(diff_matrix[i, j]):
            val = diff_matrix[i, j]
            # Choose text colors based on the background color.
            text_color = 'white' if abs(val) > 15 else 'black'
            ax.text(j, i, f'{val:+.0f}', ha='center', va='center', 
                   fontsize=11, fontweight='bold', color=text_color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.ax.tick_params(labelsize=11)
cbar.set_label('Difference (%)', fontsize=12)

# ========== Right plot: Average safety ratio by context length ==========
ax = axes[1]

# Calculate the average safety rate for Types 2-4 (levels 1-3), excluding Type 1 / level 0.
levels_for_avg = [1, 2, 3]  # Use only levels 1, 2, and 3 to calculate the average.
pg19_avg_by_ctx = {}
rel_avg_by_ctx = {}

for ctx in ctx_lengths:
    pg19_vals = [pg19_data[level].get(ctx) for level in levels_for_avg if pg19_data[level].get(ctx) is not None]
    rel_vals = [relevant_data[level].get(ctx) for level in levels_for_avg if relevant_data[level].get(ctx) is not None]
    
    if pg19_vals:
        pg19_avg_by_ctx[ctx] = np.mean(pg19_vals)
    if rel_vals:
        rel_avg_by_ctx[ctx] = np.mean(rel_vals)

x = list(pg19_avg_by_ctx.keys())
pg19_vals = [pg19_avg_by_ctx[ctx] for ctx in x]
rel_vals = [rel_avg_by_ctx.get(ctx, np.nan) for ctx in x]

width = 0.35
x_pos = np.arange(len(x))

bars1 = ax.bar(x_pos - width/2, pg19_vals, width, label='PG19 (Irrelevant)', 
               color='#2E86AB', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x_pos + width/2, rel_vals, width, label='Relevant Context', 
               color='#E94F37', edgecolor='white', linewidth=0.5)

ax.set_xlabel('Context Length', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Safety Rate (%)', fontsize=14, fontweight='bold')
ax.set_title('Average Safety Rate (Type 2−4)', fontsize=16, fontweight='bold', pad=10)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{ctx}k' for ctx in x], fontsize=12)
ax.set_ylim(30, 100)
ax.set_yticks([30, 40, 50, 60, 70, 80, 90, 100])
ax.tick_params(axis='y', labelsize=12)
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--', axis='y')

plt.tight_layout()
plt.savefig(f'context_comparison_{model_name}.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics.
print("\n" + "=" * 50)
print("Summary (Type 2, 3, 4 = Level 1, 2, 3)")
print("=" * 50)
print(f"\n{'Context':<10} {'PG19':>10} {'Relevant':>10} {'Diff':>10}")
for ctx in ctx_lengths:
    pg19_v = pg19_avg_by_ctx.get(ctx, np.nan)
    rel_v = rel_avg_by_ctx.get(ctx, np.nan)
    diff = rel_v - pg19_v
    print(f"{ctx}k{'':<8} {pg19_v:>10.1f} {rel_v:>10.1f} {diff:>+10.1f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# 1. Basic configuration.
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'font.family': 'sans-serif'
})

# Example data: Ensure all_context_lengths, pg19_data, and relevant_data already exist in the script.
levels_to_use = [0, 1, 2, 3]
type_labels = [1, 2, 3, 4]
ctx_lengths = all_context_lengths

# 2. Build the difference matrix.
diff_matrix = np.full((len(levels_to_use), len(ctx_lengths)), np.nan)
for i, level in enumerate(levels_to_use):
    for j, ctx in enumerate(ctx_lengths):
        pg19_val = pg19_data.get(level, {}).get(ctx)
        rel_val = relevant_data.get(level, {}).get(ctx)
        if pg19_val is not None and rel_val is not None:
            diff_matrix[i, j] = rel_val - pg19_val

# 3. Create the figure.
# Increase the figure width to leave visual space for the colorbar on the right.
fig, axes = plt.subplots(2, 1, figsize=(9, 9), gridspec_kw={'height_ratios': [1.2, 1]})

# ========== Upper plot: Difference heatmap ==========
ax0 = axes[0]
norm = TwoSlopeNorm(vmin=-20, vcenter=0, vmax=40)
im = ax0.imshow(diff_matrix, cmap='RdBu', aspect='auto', norm=norm)

ax0.set_title('Difference (Relevant − PG19) %', fontsize=16, fontweight='bold', pad=15)
ax0.set_ylabel('Type', fontsize=14, fontweight='bold')
ax0.set_xticks(range(len(ctx_lengths)))
ax0.set_xticklabels([f'{x}k' for x in ctx_lengths])
ax0.set_yticks(range(len(levels_to_use)))
ax0.set_yticklabels([f'Type {t}' for t in type_labels])

# Add numeric labels.
for i in range(len(levels_to_use)):
    for j in range(len(ctx_lengths)):
        if not np.isnan(diff_matrix[i, j]):
            val = diff_matrix[i, j]
            text_color = 'white' if abs(val) > 18 else 'black'
            ax0.text(j, i, f'{val:+.0f}', ha='center', va='center', 
                    fontsize=12, fontweight='bold', color=text_color)

# --- Layout adjustment: Fix the colorbar width. ---
divider = make_axes_locatable(ax0)
cax = divider.append_axes("right", size="3%", pad=0.15)
cbar = plt.colorbar(im, cax=cax)
cbar.set_label('Diff (%)', fontsize=12)

# ========== Lower plot: Average safety ratio ==========
ax1 = axes[1]
levels_for_avg = [1, 2, 3]
pg19_avg_by_ctx = {}
rel_avg_by_ctx = {}

for ctx in ctx_lengths:
    pg19_vals = [pg19_data[lvl].get(ctx) for lvl in levels_for_avg if pg19_data.get(lvl, {}).get(ctx) is not None]
    rel_vals = [relevant_data[lvl].get(ctx) for lvl in levels_for_avg if relevant_data.get(lvl, {}).get(ctx) is not None]
    if pg19_vals: pg19_avg_by_ctx[ctx] = np.mean(pg19_vals)
    if rel_vals: rel_avg_by_ctx[ctx] = np.mean(rel_vals)

x = list(pg19_avg_by_ctx.keys())
p_vals = [pg19_avg_by_ctx[ctx] for ctx in x]
r_vals = [rel_avg_by_ctx.get(ctx, np.nan) for ctx in x]

width = 0.35
x_pos = np.arange(len(x))

ax1.bar(x_pos - width/2, p_vals, width, label='PG19 (Irrelevant)', color='#2E86AB', edgecolor='white')
ax1.bar(x_pos + width/2, r_vals, width, label='Relevant Context', color='#E94F37', edgecolor='white')

ax1.set_xlabel('Context Length', fontsize=14, fontweight='bold')
ax1.set_ylabel('Avg Safety Rate (%)', fontsize=14, fontweight='bold')
ax1.set_title('Average Safety Rate (Type 2−4)', fontsize=16, fontweight='bold', pad=15)
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'{ctx}k' for ctx in x])
ax1.set_ylim(30, 105) # Raise the upper limit slightly to leave room for the legend.
ax1.grid(True, alpha=0.3, linestyle='--', axis='y')
ax1.legend(loc='upper right', framealpha=0.9)

# Adjust spacing automatically.
plt.tight_layout()

# Use the following code to adjust vertical spacing manually if needed:
# plt.subplots_adjust(hspace=0.3)

plt.savefig(f'context_comparison_{model_name}.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics, preserving the original behavior.
print("\n" + "=" * 50)
print(f"{'Context':<10} {'PG19':>10} {'Relevant':>10} {'Diff':>10}")
for ctx in ctx_lengths:
    p_v, r_v = pg19_avg_by_ctx.get(ctx, 0), rel_avg_by_ctx.get(ctx, 0)
    print(f"{ctx}k{'':<8} {p_v:>10.1f} {r_v:>10.1f} {r_v-p_v:>+10.1f}")